# Clip Validator

Generate short GIFs from your annotations so you can visually verify:

1. **Label consistency** — every clip's center frame should be the visual peak of the punch (full arm extension)
2. **Window appropriateness** — the 1s lead-up should show the punch setup; the 0.5s follow-through should show the retraction
3. **Pipeline correctness** — what goes into training is what you think is going in

## How this works

- Reads `data/annotations.csv`
- Picks a random sample of N annotations per class
- For each: opens the video, extracts a window of frames around the contact, subsamples to T frames, saves as a GIF
- Writes to `data/clip_previews/<class>/<video>_f<frame>.gif`

Then you just open the folder in Finder and scroll through.

## Run order

1. Install cell (once)
2. Imports
3. Configuration — tune window sizes and sample count
4. PyAVReader class
5. Clip extractor function
6. Generate previews
7. Summary report

## 1. Install (once)

In [6]:
# Run once if you haven't already
# !pip install av pandas Pillow numpy

## 2. Imports

In [7]:
import os
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import av

## 3. Configuration

In [8]:
# === EDIT THESE ===

CSV_PATH     = "data/annotations.csv"
VIDEO_DIR    = "data/downloaded-videos"
OUTPUT_DIR   = "data/clip_previews"

# Window definition (in frames, at the VIDEO's native fps)
# NOTE: these assume ~30fps. If your videos are 60fps, double them.
N_PRE        = 10   # frames BEFORE contact (1.0s at 30fps)
N_POST       = 5   # frames AFTER contact  (0.5s at 30fps)

# Temporal subsampling: T frames uniformly sampled from the window
T            = 16

# How many samples per class to generate (set to None to do ALL annotations)
N_PER_CLASS  = 15

# Output GIF settings
GIF_WIDTH    = 480     # resize to this width
GIF_FPS      = 12      # playback speed; lower = slower so you can see detail

# Reproducibility
RANDOM_SEED  = 42

## 4. PyAVReader (same as annotator)

In [9]:
class PyAVReader:
    """Frame-accurate random-access video reader built on PyAV."""

    def __init__(self, path, cache_size=64):
        self.path = str(path)
        self.container = av.open(self.path)
        self.stream = self.container.streams.video[0]
        self.stream.thread_type = "AUTO"

        if self.stream.average_rate is not None:
            self.fps = float(self.stream.average_rate)
        else:
            self.fps = float(self.stream.guessed_rate or 30.0)

        self.total_frames = self.stream.frames
        if not self.total_frames:
            self.total_frames = sum(1 for _ in self.container.decode(video=0))
            self.container.close()
            self.container = av.open(self.path)
            self.stream = self.container.streams.video[0]
            self.stream.thread_type = "AUTO"

        self.time_base = self.stream.time_base
        self.cache = {}
        self.cache_order = []
        self.cache_size = cache_size
        self._last_decoded_idx = -1
        self._decode_iter = None

    def _pts_for_frame(self, frame_index):
        seconds = frame_index / self.fps
        return int(seconds / float(self.time_base))

    def _cache_put(self, idx, arr):
        if idx in self.cache:
            self.cache_order.remove(idx)
        elif len(self.cache_order) >= self.cache_size:
            drop = self.cache_order.pop(0)
            self.cache.pop(drop, None)
        self.cache[idx] = arr
        self.cache_order.append(idx)

    def get_frame(self, frame_index):
        frame_index = max(0, min(int(frame_index), self.total_frames - 1))
        if frame_index in self.cache:
            return self.cache[frame_index]

        if frame_index == self._last_decoded_idx + 1 and self._decode_iter is not None:
            try:
                frame = next(self._decode_iter)
                arr = frame.to_ndarray(format="rgb24")
                self._last_decoded_idx = frame_index
                self._cache_put(frame_index, arr)
                return arr
            except StopIteration:
                self._decode_iter = None

        target_pts = self._pts_for_frame(frame_index)
        self.container.seek(target_pts, any_frame=False, backward=True, stream=self.stream)
        self._decode_iter = self.container.decode(video=0)

        last_arr = None
        current_idx = -1
        for frame in self._decode_iter:
            if frame.pts is None:
                continue
            t = float(frame.pts * self.time_base)
            current_idx = int(round(t * self.fps))
            if current_idx > frame_index:
                break
            last_arr = frame.to_ndarray(format="rgb24")
            if current_idx == frame_index:
                self._last_decoded_idx = frame_index
                self._cache_put(frame_index, last_arr)
                return last_arr

        if last_arr is not None:
            self._last_decoded_idx = current_idx
            self._cache_put(current_idx, last_arr)
            return last_arr
        raise RuntimeError(f"Could not decode frame {frame_index}")

    def __len__(self):
        return self.total_frames

    def close(self):
        self.container.close()

## 5. Clip extractor and GIF writer

In [10]:
def extract_clip(reader, contact_frame, n_pre, n_post, t_target):
    """Extract a window [contact-n_pre, contact+n_post], subsample to t_target frames.

    Returns (frames, contact_idx_in_output) where contact_idx_in_output tells you
    where the contact frame landed in the subsampled clip — useful for visual checks.
    """
    start = max(0, contact_frame - n_pre)
    end = min(len(reader), contact_frame + n_post + 1)  # exclusive
    window_indices = list(range(start, end))

    if len(window_indices) < 2:
        return None, None

    # Uniform subsample to t_target frames. np.linspace gives us evenly-spaced
    # indices INTO window_indices (not into the video).
    sample_positions = np.linspace(0, len(window_indices) - 1, t_target).round().astype(int)
    sampled_frame_indices = [window_indices[p] for p in sample_positions]

    # Figure out which subsampled index is closest to the contact frame
    # (for drawing the "contact" marker)
    contact_pos_in_window = contact_frame - start
    contact_idx_in_output = int(np.abs(sample_positions - contact_pos_in_window).argmin())

    # Pull frames. Sort indices to decode forward efficiently, then reorder.
    order = np.argsort(sampled_frame_indices)
    frames_by_order = []
    for pos in order:
        idx = sampled_frame_indices[pos]
        frames_by_order.append((pos, reader.get_frame(idx).copy()))
    frames_by_order.sort(key=lambda x: x[0])
    frames = [f for _, f in frames_by_order]

    return frames, contact_idx_in_output


def save_gif(frames, contact_idx, out_path, width, fps, label, contact_frame):
    """Save frames as an animated GIF. Highlights the contact frame with a red border."""
    pil_frames = []
    for i, arr in enumerate(frames):
        img = Image.fromarray(arr)
        # Resize
        if img.width != width:
            ratio = width / img.width
            img = img.resize((width, int(img.height * ratio)), Image.BILINEAR)
        # Mark the contact frame with a red border, mark others with a thin gray one
        if i == contact_idx:
            bordered = Image.new("RGB", (img.width + 12, img.height + 12), (220, 30, 30))
        else:
            bordered = Image.new("RGB", (img.width + 12, img.height + 12), (60, 60, 60))
        bordered.paste(img, (6, 6))
        # Add a small label strip at the top
        from PIL import ImageDraw
        draw = ImageDraw.Draw(bordered)
        tag = f"{label}  f{contact_frame}  [{i+1}/{len(frames)}]"
        if i == contact_idx:
            tag = "CONTACT  " + tag
        draw.text((10, 0), tag, fill=(255, 255, 255))
        pil_frames.append(bordered)

    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
    )

## 6. Generate preview GIFs

In [11]:
# --- Load annotations ---
csv_path = Path(CSV_PATH)
if not csv_path.exists():
    raise FileNotFoundError(f"No annotations found at {csv_path}")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} annotations across {df['video_id'].nunique()} videos")
print(df['punch_type'].value_counts().to_string())
print()

# --- Sample per class ---
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sampled_rows = []
for cls, group in df.groupby('punch_type'):
    n = min(N_PER_CLASS, len(group)) if N_PER_CLASS else len(group)
    sampled_rows.append(group.sample(n=n, random_state=RANDOM_SEED))
sampled = pd.concat(sampled_rows).reset_index(drop=True)
print(f"Selected {len(sampled)} clips to generate")
print()

# --- Ensure output dir ---
out_root = Path(OUTPUT_DIR)
out_root.mkdir(parents=True, exist_ok=True)

# --- Generate, grouped by video so we open each video only once ---
results = {"ok": 0, "too_short": 0, "missing_video": 0, "error": 0}
report_rows = []

for video_id, grp in sampled.groupby('video_id'):
    video_path = Path(VIDEO_DIR) / video_id
    if not video_path.exists():
        print(f"[SKIP] Video not found: {video_path}  ({len(grp)} annotations)")
        results["missing_video"] += len(grp)
        continue

    print(f"Opening {video_id} ({len(grp)} clips)...")
    try:
        reader = PyAVReader(video_path)
    except Exception as e:
        print(f"  Could not open: {e}")
        results["error"] += len(grp)
        continue

    for _, row in grp.iterrows():
        cls = row['punch_type']
        contact_frame = int(row['frame_index'])
        class_dir = out_root / cls
        class_dir.mkdir(parents=True, exist_ok=True)

        safe_video = video_id.replace('.mp4', '').replace('/', '_')
        out_path = class_dir / f"{safe_video}_f{contact_frame:06d}.gif"

        try:
            frames, contact_idx = extract_clip(reader, contact_frame, N_PRE, N_POST, T)
            if frames is None:
                results["too_short"] += 1
                continue
            save_gif(frames, contact_idx, out_path, GIF_WIDTH, GIF_FPS, cls, contact_frame)
            results["ok"] += 1
            report_rows.append({
                "class": cls,
                "video": video_id,
                "frame": contact_frame,
                "path": str(out_path),
                "window_start": max(0, contact_frame - N_PRE),
                "window_end": min(len(reader), contact_frame + N_POST),
                "contact_idx_in_clip": contact_idx,
            })
        except Exception as e:
            print(f"  ERROR on frame {contact_frame} ({cls}): {e}")
            results["error"] += 1

    reader.close()

print()
print("=" * 50)
print(f"  OK:            {results['ok']}")
print(f"  Too short:     {results['too_short']}")
print(f"  Missing video: {results['missing_video']}")
print(f"  Error:         {results['error']}")
print()
print(f"Output: {out_root.resolve()}")
print("Open that folder in Finder and scroll through each class subfolder.")

Loaded 540 annotations across 1 videos
punch_type
jab     310
none    140
hook     90

Selected 45 clips to generate

Opening V1.mp4 (45 clips)...

  OK:            45
  Too short:     0
  Missing video: 0
  Error:         0

Output: /Users/krishna/dev/fightflow/data/clip_previews
Open that folder in Finder and scroll through each class subfolder.


## 7. Summary report

In [ ]:
# Quick sanity report on the generated clips
if not report_rows:
    print("No clips generated. Fix any errors above and re-run.")
else:
    rep = pd.DataFrame(report_rows)
    print(f"Generated {len(rep)} clips")
    print()
    print("Per class counts:")
    print(rep['class'].value_counts().to_string())
    print()
    print("Contact frame position in subsampled clip (should be consistent across clips):")
    print(rep.groupby('class')['contact_idx_in_clip'].describe()[['mean', 'std', 'min', 'max']].to_string())
    print()
    print(f"If you used N_PRE={N_PRE}, N_POST={N_POST}, T={T},")
    print(f"the contact should land at approximately index {round(N_PRE / (N_PRE + N_POST) * (T - 1))} in most clips.")
    print()
    print("Open this folder to view GIFs:")
    print(f"  {Path(OUTPUT_DIR).resolve()}")

## What to look for when reviewing

Open `data/clip_previews/` in Finder. Browse each class folder and scroll through the GIFs. You're looking for:

### Label consistency
- Every **jab** GIF should show a straight lead-hand punch
- Every **cross** should show a rear-hand straight punch crossing the centerline
- Every **hook** should show a looping horizontal punch
- Every **uppercut** should show a rising vertical punch
- Any outliers? Note the filename and mark them as questionable — you might relabel or drop them.

### Contact frame is the peak
The CONTACT frame (red border) should be the visual peak — full arm extension, maximum reach. If it looks like the peak is 1-2 frames before or after the highlighted one, that's normal subsampling jitter. If it's consistently 3+ frames off, your labeling definition drifted.

### Window framing feels right
- Can you see the setup in the lead-up frames? If the GIF starts mid-punch, `N_PRE` is too small.
- Does the GIF end before the retraction completes? If so, `N_POST` is too small.
- Does the GIF have lots of "nothing" at the start? `N_PRE` is too big — try 20-25 frames instead of 30.

### If you need to retune
Just change `N_PRE` / `N_POST` / `T` in the config cell and re-run. The extraction is fast.

## After validation

Once you're happy with how the clips look, the SAME extraction logic becomes your training data loader — you'll call `extract_clip()` inside a PyTorch Dataset's `__getitem__`, return the frames as a tensor instead of saving a GIF. No re-annotation needed, no pipeline differences between "validation" and "training."